# Aula 15 - Notebook: Simulação de Injeção de Falhas e Desvio Automático de Fluxo

Neste notebook simulamos uma contingência em tempo real: o rompimento da tubulação direta do reator com acionamento do detector de gás e recálculo dinâmico de rota pelo SCADA.


In [1]:
def formatar_tabela(dados):
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz, rotulos_linhas, rotulos_cols):
    """Formata matriz 2D em tabela ASCII pura."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

class GrafoTubulacao:
    def __init__(self, vertices):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n): self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes = []

    def adicionar_tubulacao(self, origem, destino, comprimento_m, tag_valvula, diametro_pol=4.0):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        self.arestas_detalhes.append({
            "Origem": origem, "Destino": destino,
            "Comprimento (m)": comprimento_m, "Válvula ISA": tag_valvula, "Diâmetro (pol)": diametro_pol
        })

def criar_rede_padrao():
    nos = ["TK-301_NH3", "TK-302_H3PO4", "MAN-101", "P-101", "P-102", "R-101", "TK-303_Pulmao", "GRAN-201"]
    g = GrafoTubulacao(nos)
    g.adicionar_tubulacao("TK-301_NH3", "MAN-101", 15.0, "XV-301", 3.0)
    g.adicionar_tubulacao("TK-302_H3PO4", "MAN-101", 12.0, "XV-302", 4.0)
    g.adicionar_tubulacao("MAN-101", "P-101", 8.0, "XV-101A", 4.0)
    g.adicionar_tubulacao("MAN-101", "P-102", 10.0, "XV-101B", 4.0)
    g.adicionar_tubulacao("P-101", "R-101", 25.0, "XV-102A", 4.0)
    g.adicionar_tubulacao("P-102", "R-101", 22.0, "XV-102B", 4.0)
    g.adicionar_tubulacao("R-101", "GRAN-201", 30.0, "XV-201", 6.0)
    g.adicionar_tubulacao("R-101", "TK-303_Pulmao", 18.0, "XV-202", 6.0)
    g.adicionar_tubulacao("TK-303_Pulmao", "GRAN-201", 20.0, "XV-203", 6.0)
    return g

import time
import heapq
from typing import Dict, Any, Tuple, Optional, Set, List

rede = criar_rede_padrao()

class RoteadorDijkstra:
    def __init__(self, grafo: GrafoTubulacao):
        self.g = grafo

    def calcular_menor_caminho(self, origem: str, destino: str, 
                               bloqueios: Optional[Set[str]] = None) -> Tuple[float, List[str]]:
        if bloqueios is None: bloqueios = set()
        if origem in bloqueios or destino in bloqueios: return float('inf'), []
        dist = {v: float('inf') for v in self.g.vertices}
        pred = {v: None for v in self.g.vertices}
        dist[origem] = 0.0
        heap = [(0.0, origem)]
        while heap:
            d_u, u = heapq.heappop(heap)
            if d_u > dist[u]: continue
            if u == destino: break
            u_idx = self.g.v_to_idx[u]
            for v_idx in range(self.g.n):
                v = self.g.idx_to_v[v_idx]
                peso = self.g.adj_pesos[u_idx][v_idx]
                if peso < float('inf') and v not in bloqueios:
                    nova_d = d_u + peso
                    if nova_d < dist[v]:
                        dist[v] = nova_d
                        pred[v] = u
                        heapq.heappush(heap, (nova_d, v))
        caminho = []
        atual = destino
        while atual is not None:
            caminho.append(atual)
            atual = pred[atual]
        caminho.reverse()
        if caminho and caminho[0] == origem:
            return dist[destino], caminho
        return float('inf'), []

class SistemaDesvioAutomatico:
    def __init__(self, grafo: GrafoTubulacao):
        self.grafo = grafo
        self.roteador = RoteadorDijkstra(grafo)
        
    def tratar_evento_vazamento(self, origem_vaz: str, destino_vaz: str,
                                origem_fluxo: str, destino_fluxo: str) -> Dict[str, Any]:
        t0 = time.perf_counter()
        u = self.grafo.v_to_idx[origem_vaz]
        v = self.grafo.v_to_idx[destino_vaz]
        self.grafo.adj_pesos[u][v] = float('inf')
        self.grafo.adj_binaria[u][v] = 0
        
        novo_custo, nova_rota = self.roteador.calcular_menor_caminho(origem_fluxo, destino_fluxo)
        t_ms = (time.perf_counter() - t0) * 1000.0
        
        return {
            "Trecho_Isolado": f"{origem_vaz} -> {destino_vaz}",
            "Nova_Rota_Ativa": " -> ".join(nova_rota),
            "Comprimento_Total_m": f"{novo_custo:.1f}",
            "Tempo_Decisão_ms": f"{t_ms:.4f}"
        }

desviador = SistemaDesvioAutomatico(rede)
res_vaz = desviador.tratar_evento_vazamento("R-101", "GRAN-201", "TK-301_NH3", "GRAN-201")
print("=== RELATÓRIO DE DESVIO AUTOMÁTICO DE FLUIDOS ===")
print(formatar_tabela([res_vaz]))
assert "TK-303_Pulmao" in res_vaz["Nova_Rota_Ativa"]


=== RELATÓRIO DE DESVIO AUTOMÁTICO DE FLUIDOS ===
Trecho_Isolado    | Nova_Rota_Ativa                                                      | Comprimento_Total_m | Tempo_Decisão_ms
------------------+----------------------------------------------------------------------+---------------------+-----------------
R-101 -> GRAN-201 | TK-301_NH3 -> MAN-101 -> P-102 -> R-101 -> TK-303_Pulmao -> GRAN-201 | 85.0                | 0.0460          
